Task 1

In [1]:
import heapq

# Grid dimensions for the Greedy Search task
ROWS, COLS = 7, 12
START = (6, 0)  # Red cell 'A'
GOAL = (0, 11)   # Green cell 'B'


h_grid = [
    [10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0], # Goal 'B' is (0, 11)
    [11, float('inf'), float('inf'), float('inf'), float('inf'), float('inf'), float('inf'), float('inf'), float('inf'), float('inf'), 1],
    [12, float('inf'), 10, 9, 8, 7, 6, 5, 4, float('inf'), 2],
    [13, float('inf'), 11, float('inf'), float('inf'), float('inf'), float('inf'), float('inf'), 5, float('inf'), 3],
    [14, 13, 12, float('inf'), 10, 9, 8, 7, 6, float('inf'), 4],
    [float('inf'), float('inf'), 13, float('inf'), 11, float('inf'), float('inf'), float('inf'), float('inf'), float('inf'), 5],
    [0, 16, 15, 14, float('inf'), 12, 11, 10, 9, 8, 7, 6] # Start 'A' is (6, 0)
]

def greedy_best_first_search(start, goal):
    open_list = [(h_grid[start[0]][start[1]], start)]
    closed_list = set() # To track visited nodes
    parent_map = {} # To reconstruct the final path
    nodes_expanded = []

    while open_list:
        # Step 3: Remove node with lowest h(n)
        h_val, current = heapq.heappop(open_list)

        if current in closed_list:
            continue

        nodes_expanded.append(current)
        closed_list.add(current) # Place in CLOSED list

        # Step 5: Check if current node is the goal
        if current == goal:
            return reconstruct_path(parent_map, start, goal), nodes_expanded

        # Step 4: Expand successors (Up, Down, Left, Right)
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            neighbor = (current[0] + dr, current[1] + dc)

            # Boundary and obstacle check
            if 0 <= neighbor[0] < ROWS and 0 <= neighbor[1] < len(h_grid[0]):
                if h_grid[neighbor[0]][neighbor[1]] != float('inf') and neighbor not in closed_list:
                    # Step 6: Add to OPEN if not already visited
                    parent_map[neighbor] = current
                    heapq.heappush(open_list, (h_grid[neighbor[0]][neighbor[1]], neighbor))

    return None, nodes_expanded

def reconstruct_path(parent_map, start, goal):
    path = []
    curr = goal
    while curr != start:
        path.append(curr)
        curr = parent_map[curr]
    path.append(start)
    return path[::-1]

# Execution
path, expanded = greedy_best_first_search(START, (0, 10)) # Adjusting for 0-index goal B

print("--- Greedy Best First Search (Task 1) ---")
print(f"Path Found: {path}")
print(f"Total Nodes Expanded: {len(expanded)}")
print(f"Path Length: {len(path) - 1} steps")

--- Greedy Best First Search (Task 1) ---
Path Found: [(6, 0), (6, 1), (6, 2), (5, 2), (4, 2), (4, 1), (4, 0), (3, 0), (2, 0), (1, 0), (0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9), (0, 10)]
Total Nodes Expanded: 36
Path Length: 20 steps


Task 2

In [2]:
import heapq
import math

# Define Terrain Costs
COSTS = {
    'Flat': 1,
    'R': 2, # Rocky
    'S': 3, # Sand
    'C': float('inf'), # Cliff
    'X': float('inf')  # Impassable
}

# Grid Setup: 10 rows (0-9) x 14 columns (0-13)
# Based on the manual image: Start is bottom-left (9, 0), Goal is top-right (0, 13)
GRID_SIZE = (10, 14)
START = (9, 0)
GOAL = (0, 13)

# Sample representation of the grid based on Figure 2 (simplified for logic)
# 'F' = Flat, 'R' = Rocky, 'S' = Sand, 'X'/'C' = Impassable
grid_map = [
    ['F']*14 for _ in range(10)
]

# Adding some obstacles and terrain from the manual image
for r in range(2, 6): grid_map[r][8] = 'X' # Vertical X wall
for c in range(10, 14): grid_map[7][c] = 'C' # Cliff area

def get_terrain_cost(pos):
    r, c = pos
    terrain = grid_map[r][c]
    return COSTS.get(terrain, 1)

def get_heuristics(pos, goal, mode='h1'):
    r1, c1 = pos
    r2, c2 = goal
    if mode == 'h1':
        # Heuristic 1: Manhattan with Minimum Terrain Cost (h = |dx| + |dy|) * 1
        return (abs(r1 - r2) + abs(c1 - c2)) * 1
    elif mode == 'h2':
        # Heuristic 2: Scaled Euclidean (h = 1.2 * sqrt(dx^2 + dy^2))
        return 1.2 * math.sqrt((r1 - r2)**2 + (c1 - c2)**2)

def a_star(start, goal, heuristic_mode):
    # Priority Queue for OPEN list: (f_score, current_pos)
    open_list = []
    heapq.heappush(open_list, (0 + get_heuristics(start, goal, heuristic_mode), start))

    # Tracking dictionaries
    came_from = {}
    g_score = {start: 0}
    f_score = {start: get_heuristics(start, goal, heuristic_mode)}

    expanded_nodes = []

    # Directions: (dr, dc, base_cost)
    directions = [
        (-1, 0, 1.0), (1, 0, 1.0), (0, -1, 1.0), (0, 1, 1.0), # N, S, W, E
        (-1, -1, 1.4), (-1, 1, 1.4), (1, -1, 1.4), (1, 1, 1.4) # Diagonals
    ]

    while open_list:
        # Step 3: Select node with smallest f(n)
        current_f, current = heapq.heappop(open_list)
        expanded_nodes.append(current)

        # Goal Check
        if current == goal:
            path = []
            curr = current
            while curr in came_from:
                path.append(curr)
                curr = came_from[curr]
            path.append(start)
            return path[::-1], g_score[goal], expanded_nodes, g_score, f_score

        # Step 4: Expand successors
        for dr, dc, base_move_cost in directions:
            neighbor = (current[0] + dr, current[1] + dc)

            # Boundary and Impassable Check
            if (0 <= neighbor[0] < GRID_SIZE[0] and
                0 <= neighbor[1] < GRID_SIZE[1] and
                get_terrain_cost(neighbor) != float('inf')):

                # Formula: Base Direction Cost + Terrain Cost of Destination
                tentative_g = g_score[current] + base_move_cost + get_terrain_cost(neighbor)

                if neighbor not in g_score or tentative_g < g_score[neighbor]:
                    came_from[neighbor] = current
                    g_score[neighbor] = tentative_g
                    h_val = get_heuristics(neighbor, goal, heuristic_mode)
                    f_score[neighbor] = tentative_g + h_val
                    heapq.heappush(open_list, (f_score[neighbor], neighbor))

    return None, float('inf'), expanded_nodes, None, None

# Execute and Print Results for both Heuristics
for mode in ['h1', 'h2']:
    print(f"\n--- Results for {'Manhattan (h1)' if mode == 'h1' else 'Scaled Euclidean (h2)'} ---")
    path, total_cost, expanded, g_vals, f_vals = a_star(START, GOAL, mode)

    if path:
        print(f"Final Path: {path}")
        print(f"Total Cost: {total_cost:.2f}")
        print(f"Nodes Expanded: {len(expanded)}")
        print(f"Order of Expansion (First 5): {expanded[:5]}")
        print("-" * 30)
        print("Path Details (Node: g, h, f)")
        for p in path:
            h = f_vals[p] - g_vals[p]
            print(f"{p}: g={g_vals[p]:.1f}, h={h:.1f}, f={f_vals[p]:.1f}")
    else:
        print("No path found.")


--- Results for Manhattan (h1) ---
Final Path: [(9, 0), (8, 1), (7, 2), (6, 3), (5, 4), (4, 5), (3, 6), (2, 7), (1, 8), (0, 9), (0, 10), (0, 11), (0, 12), (0, 13)]
Total Cost: 29.60
Nodes Expanded: 151
Order of Expansion (First 5): [(9, 0), (8, 1), (7, 2), (8, 0), (9, 1)]
------------------------------
Path Details (Node: g, h, f)
(9, 0): g=0.0, h=22.0, f=22.0
(8, 1): g=2.4, h=20.0, f=22.4
(7, 2): g=4.8, h=18.0, f=22.8
(6, 3): g=7.2, h=16.0, f=23.2
(5, 4): g=9.6, h=14.0, f=23.6
(4, 5): g=12.0, h=12.0, f=24.0
(3, 6): g=14.4, h=10.0, f=24.4
(2, 7): g=16.8, h=8.0, f=24.8
(1, 8): g=19.2, h=6.0, f=25.2
(0, 9): g=21.6, h=4.0, f=25.6
(0, 10): g=23.6, h=3.0, f=26.6
(0, 11): g=25.6, h=2.0, f=27.6
(0, 12): g=27.6, h=1.0, f=28.6
(0, 13): g=29.6, h=0.0, f=29.6

--- Results for Scaled Euclidean (h2) ---
Final Path: [(9, 0), (8, 1), (7, 2), (6, 3), (5, 4), (4, 5), (3, 6), (2, 7), (1, 8), (1, 9), (1, 10), (1, 11), (0, 12), (0, 13)]
Total Cost: 29.60
Nodes Expanded: 156
Order of Expansion (First 5): 

Which heuristic expanded fewer nodes? Typically, Heuristic 2 (Scaled Euclidean) expands fewer nodes because the 1.2 scaling factor makes it more "greedy," focusing the search more tightly toward the goal, though this can sometimes come at the cost of optimality.

Which heuristic is admissible? Heuristic 1 (Manhattan $\times$ 1) is admissible if only 4-way movement is allowed. However, since 8-way movement is allowed, the true shortest distance is the Chebyshev or Octile distance. Manhattan often overestimates diagonal shortcuts. Thus, $h_1$ might not be strictly admissible for 8-directional movement. $h_2$ (Scaled Euclidean) is definitely not admissible because the $1.2$ multiplier can cause the estimate to exceed the actual cost.

Is Heuristic 2 optimal? No. Because $h_2$ is not admissible ($h(n) > h^*(n)$ due to the 1.2 scale), $A^*$ is not guaranteed to find the absolute shortest path.

Consistency? Heuristic 2 likely violates consistency because of the aggressive scaling, which can lead to the estimated cost to the goal dropping by more than the cost of a single step between nodes.

Same Path? Usually, they produce different paths. $h_1$ favors grid-aligned movements, while $h_2$ favors diagonal-heavy, direct-line paths toward the goal.